In [ ]:
!pip install -q transformers accelerate "lm-eval[api]"
!pip install -q immutabledict langdetect nltk absl-py


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 5.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 135.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.0/128.0 kB 17.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 49.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
import torch


In [ ]:
print(torch.cuda.get_device_name(0))

NVIDIA A100-SXM4-80GB


In [ ]:
OUT = "/content/results"
DRIVE = "/content/drive/MyDrive/dissertation/results"

In [ ]:
import subprocess, time
from datetime import datetime

def log(msg):
    print(f"[{datetime.now():%H:%M:%S}] {msg}", flush=True)

def run_evals(model, tag, revision="main"):
    base = ["lm_eval", "--model", "hf",
            "--model_args", f"pretrained={model},dtype=float16,revision={revision}",
            "--batch_size", "16", "--num_fewshot", "0", "--log_samples"]

    jobs = [
        ("arc_hella", ["--tasks", "arc_challenge,hellaswag"]),
        ("mmlu",      ["--tasks", "mmlu"]),
        ("gen",       ["--tasks", "gsm8k,ifeval", "--apply_chat_template"]),
    ]

    status = {}
    for i, (name, extra) in enumerate(jobs, 1):
        path = f"{OUT}/{tag}_{name}"
        log(f"[{i}/{len(jobs)}] START {name}")
        t0 = time.time()
        r = subprocess.run(base + extra + ["--output_path", path])
        mins = (time.time() - t0) / 60

        if r.returncode == 0:
            log(f"[{i}/{len(jobs)}] PASS  {name}  ({mins:.1f} min)")
            status[name] = "pass"
        else:
            log(f"[{i}/{len(jobs)}] FAIL  {name}  ({mins:.1f} min, exit {r.returncode})")
            status[name] = f"fail (exit {r.returncode})"

    log(f"summary: {status}")
    return status

In [ ]:
import json, random
from datasets import load_dataset

SEED = 0
N = 20

ds = load_dataset("openai/gsm8k", "main", split="test")

rng = random.Random(SEED)
idx = rng.sample(range(len(ds)), N)
prompts = [ds[i]["question"] for i in idx]

json.dump({"seed": SEED, "n": N, "source": "gsm8k/main/test",
           "indices": idx, "prompts": prompts},
          open("latency_prompts.json", "w"), indent=2)

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [ ]:
import torch, time, json
from transformers import AutoModelForCausalLM, AutoTokenizer

def measure_efficiency(model_id, prompts):
    torch.cuda.reset_peak_memory_stats()
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, dtype=torch.float16, device_map="auto"
    ).eval()

    # weights in memory, the best single predictor of decode speed
    weights_gb = sum(p.numel() * p.element_size() for p in model.parameters()) / 1e9

    # prefill FLOPs, one forward pass over a single prompt
    flops_prefill_per_token = 2 * sum(p.numel() for p in model.parameters())

    for p in prompts[:2]:                                    # warm-up, discarded
        model.generate(**tok(p, return_tensors="pt").to(model.device), max_new_tokens=16)

    ttft, tps = [], []
    for p in prompts:
        inp = tok(p, return_tensors="pt").to(model.device)

        torch.cuda.synchronize(); t0 = time.perf_counter()
        model.generate(**inp, max_new_tokens=1)
        torch.cuda.synchronize(); ttft.append(time.perf_counter() - t0)

        torch.cuda.synchronize(); t0 = time.perf_counter()
        out = model.generate(**inp, max_new_tokens=128, do_sample=False)
        torch.cuda.synchronize(); dt = time.perf_counter() - t0
        tps.append((out.shape[1] - inp.input_ids.shape[1]) / dt)

    return {
        "model": model_id,
        "ttft_s": sum(ttft) / len(ttft),
        "tok_per_s": sum(tps) / len(tps),
        "peak_mem_gb": torch.cuda.max_memory_allocated() / 1e9,
        "weights_gb": weights_gb,
        "flops_prefill_per_token": flops_prefill_per_token,
    }

In [ ]:
import json, glob, math
from collections import defaultdict

def load_samples(path):
    """Yield (probs, correct_index) per question."""
    with open(path) as f:
        for line in f:
            row = json.loads(line)
            logprobs = [float(r[0]) for r in row["filtered_resps"]]
            m = max(logprobs)
            exps = [math.exp(lp - m) for lp in logprobs]      # shift for stability
            total = sum(exps)
            probs = [e / total for e in exps]
            yield probs, int(row["target"])

def compute_ece(paths, n_bins=10):
    conf, correct = [], []
    for p in paths:
        for probs, target in load_samples(p):
            pred = max(range(len(probs)), key=lambda i: probs[i])
            conf.append(probs[pred])
            correct.append(1.0 if pred == target else 0.0)

    n = len(conf)
    bins = defaultdict(list)
    for c, y in zip(conf, correct):
        b = min(int(c * n_bins), n_bins - 1)
        bins[b].append((c, y))

    ece = 0.0
    table = []
    for b in sorted(bins):
        items = bins[b]
        mean_conf = sum(c for c, _ in items) / len(items)
        mean_acc = sum(y for _, y in items) / len(items)
        ece += (len(items) / n) * abs(mean_conf - mean_acc)
        table.append({
            "bin": f"{b/n_bins:.1f}-{(b+1)/n_bins:.1f}",
            "n": len(items),
            "mean_conf": round(mean_conf, 4),
            "mean_acc": round(mean_acc, 4),
            "gap": round(mean_conf - mean_acc, 4),
        })

    return {
        "ece": ece,
        "n": n,
        "mean_conf": sum(conf) / n,
        "accuracy": sum(correct) / n,
        "bins": table,
    }

In [ ]:
import json, os, torch, transformers
from datetime import datetime

OUT = "/content/drive/MyDrive/dissertation/results"

def pipeline(model_id, tag, prompts, outdir=OUT):
    os.makedirs(outdir, exist_ok=True)

    run_evals(model_id, tag)                          # writes its own JSONs
    eff = measure_efficiency(model_id, prompts)
    cal = compute_calibration(tag)                    # stub for now

    row = {
        **eff,
        **cal,
        "tag": tag,
        "timestamp": datetime.now().isoformat(),
        "env": {
            "transformers": transformers.__version__,
            "torch": torch.__version__,
            "gpu": torch.cuda.get_device_name(0),
        },
    }

    with open(f"{outdir}/{tag}_summary.json", "w") as f:
        json.dump(row, f, indent=2)

    return row

In [ ]:
prompts = json.load(open("latency_prompts.json"))["prompts"]

for mid, tag in [("Qwen/Qwen2.5-0.5B-Instruct", "q05b"),("Qwen/Qwen2.5-1.5B-Instruct", "q15b"),("Qwen/Qwen2.5-7B-Instruct", "q07b")]:
    pipeline(mid, tag, prompts)

[16:14:21] [1/3] START arc_hella
[16:17:48] [1/3] PASS  arc_hella  (3.5 min)
[16:17:48] [2/3] START mmlu
[16:26:44] [2/3] PASS  mmlu  (8.9 min)
[16:26:44] [3/3] START gen
[16:55:34] [3/3] PASS  gen  (28.8 min)
[16:55:34] summary: {'arc_hella': 'pass', 'mmlu': 'pass', 'gen': 'pass'}


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[16:56:51] [1/3] START arc_hella
[17:00:17] [1/3] PASS  arc_hella  (3.4 min)
[17:00:17] [2/3] START mmlu
[17:04:30] [2/3] PASS  mmlu  (4.2 min)
[17:04:30] [3/3] START gen
[17:35:54] [3/3] PASS  gen  (31.4 min)
[17:35:54] summary: {'arc_hella': 'pass', 'mmlu': 'pass', 'gen': 'pass'}


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[17:37:24] [1/3] START arc_hella
[17:44:06] [1/3] PASS  arc_hella  (6.7 min)
[17:44:06] [2/3] START mmlu
[17:49:51] [2/3] PASS  mmlu  (5.7 min)
[17:49:51] [3/3] START gen
[18:22:12] [3/3] PASS  gen  (32.4 min)
[18:22:12] summary: {'arc_hella': 'pass', 'mmlu': 'pass', 'gen': 'pass'}


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [ ]:
!zip -r results_backup.zip /content/drive/MyDrive/dissertation/results

from google.colab import files
files.download('results_backup.zip')

  adding: content/drive/MyDrive/dissertation/results/ (stored 0%)
  adding: content/drive/MyDrive/dissertation/results/q05b_arc_hella/ (stored 0%)
  adding: content/drive/MyDrive/dissertation/results/q05b_arc_hella/Qwen__Qwen2.5-0.5B-Instruct/ (stored 0%)
  adding: content/drive/MyDrive/dissertation/results/q05b_arc_hella/Qwen__Qwen2.5-0.5B-Instruct/samples_arc_challenge_2026-08-22T16-17-44.283237.jsonl (deflated 84%)
  adding: content/drive/MyDrive/dissertation/results/q05b_arc_hella/Qwen__Qwen2.5-0.5B-Instruct/results_2026-08-22T16-17-44.283237.json (deflated 69%)
  adding: content/drive/MyDrive/dissertation/results/q05b_arc_hella/Qwen__Qwen2.5-0.5B-Instruct/samples_hellaswag_2026-08-22T16-17-44.283237.jsonl (deflated 86%)
  adding: content/drive/MyDrive/dissertation/results/q07b_arc_hella/ (stored 0%)
  adding: content/drive/MyDrive/dissertation/results/q07b_arc_hella/Qwen__Qwen2.5-7B-Instruct/ (stored 0%)
  adding: content/drive/MyDrive/dissertation/results/q07b_arc_hella/Qwen__Qwe

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import json, glob

files = glob.glob("/content/drive/MyDrive/dissertation/results/**/samples_arc_challenge*.jsonl", recursive=True)
print(len(files), "files found")
print(files[0] if files else "none")

3 files found
/content/drive/MyDrive/dissertation/results/Qwen 0.5 B/q05b_arc_hella/Qwen__Qwen2.5-0.5B-Instruct/samples_arc_challenge_2026-08-22T16-17-44.283237.jsonl


In [3]:
row = json.loads(open(files[0]).readline())
print(list(row.keys()))
print("filtered_resps:", row.get("filtered_resps"))
print("target:", row.get("target"))

['doc_id', 'doc', 'target', 'arguments', 'resps', 'filtered_resps', 'filter', 'metrics', 'doc_hash', 'prompt_hash', 'target_hash', 'acc', 'acc_norm']
filtered_resps: [['-24.6875', 'False'], ['-28.078125', 'False'], ['-24.828125', 'False'], ['-25.015625', 'False']]
target: 2


In [4]:
import json, glob, math
from collections import defaultdict

def load_samples(path):
    with open(path) as f:
        for line in f:
            row = json.loads(line)
            logprobs = [float(r[0]) for r in row["filtered_resps"]]
            m = max(logprobs)
            exps = [math.exp(lp - m) for lp in logprobs]
            total = sum(exps)
            probs = [e / total for e in exps]
            yield probs, int(row["target"])

def compute_ece(paths, n_bins=10):
    conf, correct = [], []
    for p in paths:
        for probs, target in load_samples(p):
            pred = max(range(len(probs)), key=lambda i: probs[i])
            conf.append(probs[pred])
            correct.append(1.0 if pred == target else 0.0)

    n = len(conf)
    bins = defaultdict(list)
    for c, y in zip(conf, correct):
        b = min(int(c * n_bins), n_bins - 1)
        bins[b].append((c, y))

    ece = 0.0
    table = []
    for b in sorted(bins):
        items = bins[b]
        mean_conf = sum(c for c, _ in items) / len(items)
        mean_acc = sum(y for _, y in items) / len(items)
        ece += (len(items) / n) * abs(mean_conf - mean_acc)
        table.append({
            "bin": f"{b/n_bins:.1f}-{(b+1)/n_bins:.1f}",
            "n": len(items),
            "mean_conf": round(mean_conf, 4),
            "mean_acc": round(mean_acc, 4),
            "gap": round(mean_conf - mean_acc, 4),
        })

    return {"ece": ece, "n": n,
            "mean_conf": sum(conf)/n,
            "accuracy": sum(correct)/n,
            "bins": table}

In [6]:
BASE = "/content/drive/MyDrive/dissertation/results"

for f in sorted(glob.glob(f"{BASE}/**/samples_arc_challenge*.jsonl", recursive=True)):
    print(f)

/content/drive/MyDrive/dissertation/results/Qwen 0.5 B/q05b_arc_hella/Qwen__Qwen2.5-0.5B-Instruct/samples_arc_challenge_2026-08-22T16-17-44.283237.jsonl
/content/drive/MyDrive/dissertation/results/Qwen 1.5 B/q15b_arc_hella/Qwen__Qwen2.5-1.5B-Instruct/samples_arc_challenge_2026-08-22T17-00-15.445057.jsonl
/content/drive/MyDrive/dissertation/results/Qwen 7B/q07b_arc_hella/Qwen__Qwen2.5-7B-Instruct/samples_arc_challenge_2026-08-22T17-44-04.074264.jsonl


In [7]:
FOLDERS = {"q05b": "Qwen 0.5 B", "q15b": "Qwen 1.5 B", "q07b": "Qwen 7B"}

for tag, folder in FOLDERS.items():
    for task in ["arc_challenge", "hellaswag", "mmlu"]:
        paths = glob.glob(f"{BASE}/{folder}/**/samples_{task}*.jsonl", recursive=True)
        if not paths:
            print(f"{tag:6} {task:15} no files")
            continue
        r = compute_ece(paths)
        print(f"{tag:6} {task:15} ECE {r['ece']:.4f}  acc {r['accuracy']:.4f}  conf {r['mean_conf']:.4f}  n={r['n']}")

q05b   arc_challenge   ECE 0.5004  acc 0.3055  conf 0.8058  n=1172
q05b   hellaswag       ECE 0.5570  acc 0.4063  conf 0.9633  n=10042
q05b   mmlu            ECE 0.2251  acc 0.4583  conf 0.6834  n=14042
q15b   arc_challenge   ECE 0.3834  acc 0.4352  conf 0.8186  n=1172
q15b   hellaswag       ECE 0.4579  acc 0.5081  conf 0.9660  n=10042
q15b   mmlu            ECE 0.1989  acc 0.6013  conf 0.8002  n=14042
q07b   arc_challenge   ECE 0.3595  acc 0.5265  conf 0.8860  n=1172
q07b   hellaswag       ECE 0.3513  acc 0.6202  conf 0.9715  n=10042
q07b   mmlu            ECE 0.2118  acc 0.7179  conf 0.9294  n=14042


In [1]:

!pip install -q "lm-eval[api]" transformers accelerate bitsandbytes
!pip install -U bitsandbytes
!python -c "import bitsandbytes; print(bitsandbytes.__version__)"

0.50.2


In [2]:
import torch, gc, json, os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from lm_eval.models.huggingface import HFLM
import lm_eval

MID = "Qwen/Qwen2.5-0.5B-Instruct"
SHA = "7ae557604adf67be50417f59c2c2f167def9a775"
TASK = "arc_challenge"

def build(method):
    tok = AutoTokenizer.from_pretrained(MID, revision=SHA)
    if method == "baseline":
        m = AutoModelForCausalLM.from_pretrained(
            MID, revision=SHA, dtype=torch.float16, device_map="auto")
    elif method == "nf4":
        cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
        )
        m = AutoModelForCausalLM.from_pretrained(
            MID, revision=SHA, quantization_config=cfg, device_map="auto")
    else:
        raise ValueError(method)
    return m.eval(), tok

results = {}

for method in ["baseline", "nf4"]:
    print(f"\n=== {method} ===")
    model, tok = build(method)

    weights_gb = sum(p.numel() * p.element_size() for p in model.parameters()) / 1e9
    print(f"weights: {weights_gb:.3f} GB")

    lm = HFLM(pretrained=model, tokenizer=tok, batch_size=16)
    res = lm_eval.simple_evaluate(model=lm, tasks=[TASK], num_fewshot=0)
    r = res["results"][TASK]

    results[method] = {
        "weights_gb": weights_gb,
        "acc": r["acc,none"],
        "acc_stderr": r["acc_stderr,none"],
        "acc_norm": r["acc_norm,none"],
        "acc_norm_stderr": r["acc_norm_stderr,none"],
    }
    print(f"acc {r['acc,none']:.4f}   acc_norm {r['acc_norm,none']:.4f}")

    del model, lm
    gc.collect()
    torch.cuda.empty_cache()

print("\n=== comparison ===")
b, q = results["baseline"], results["nf4"]
print(f"weights   {b['weights_gb']:.3f} -> {q['weights_gb']:.3f} GB  ({q['weights_gb']/b['weights_gb']:.2f}x)")
print(f"acc_norm  {b['acc_norm']:.4f} -> {q['acc_norm']:.4f}  (delta {q['acc_norm']-b['acc_norm']:+.4f})")
print(f"stderr    +/- {b['acc_norm_stderr']:.4f}")


=== baseline ===


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

weights: 0.988 GB


Running loglikelihood requests: 100%|██████████| 4687/4687 [00:10<00:00, 449.72it/s]


acc 0.3055   acc_norm 0.3379

=== nf4 ===


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

weights: 0.451 GB


Running loglikelihood requests: 100%|██████████| 4687/4687 [00:18<00:00, 259.30it/s]


acc 0.2918   acc_norm 0.3259

=== comparison ===
weights   0.988 -> 0.451 GB  (0.46x)
acc_norm  0.3379 -> 0.3259  (delta -0.0119)
stderr    +/- 0.0138


In [5]:
model, tok = build("nf4")

for n, p in model.named_parameters():
    print(n, tuple(p.shape), p.dtype, f"{p.numel()*p.element_size()/1e6:.1f} MB")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

model.embed_tokens.weight (151936, 896) torch.bfloat16 272.3 MB
model.layers.0.self_attn.q_proj.weight (401408, 1) torch.uint8 0.4 MB
model.layers.0.self_attn.q_proj.bias (896,) torch.bfloat16 0.0 MB
model.layers.0.self_attn.k_proj.weight (57344, 1) torch.uint8 0.1 MB
model.layers.0.self_attn.k_proj.bias (128,) torch.bfloat16 0.0 MB
model.layers.0.self_attn.v_proj.weight (57344, 1) torch.uint8 0.1 MB
model.layers.0.self_attn.v_proj.bias (128,) torch.bfloat16 0.0 MB
model.layers.0.self_attn.o_proj.weight (401408, 1) torch.uint8 0.4 MB
model.layers.0.mlp.gate_proj.weight (2179072, 1) torch.uint8 2.2 MB
model.layers.0.mlp.up_proj.weight (2179072, 1) torch.uint8 2.2 MB
model.layers.0.mlp.down_proj.weight (2179072, 1) torch.uint8 2.2 MB
model.layers.0.input_layernorm.weight (896,) torch.bfloat16 0.0 MB
model.layers.0.post_attention_layernorm.weight (896,) torch.bfloat16 0.0 MB
model.layers.1.self_attn.q_proj.weight (401408, 1) torch.uint8 0.4 MB
model.layers.1.self_attn.q_proj.bias (896,) t